
Домашняя Работа 2. Онищенко Геннадий 336883.

Особых трудностей не возникло.

Буду рад обратной связи, спасибо!





# 1. Считать датасет из файла train.csv

1. Сначала необходимо импортировать библиотеку pandas, как на лекции.

In [1]:
import pandas as pd

2. Теперь мы считываем датасет из файла `train.csv`





In [2]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/train.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Здесь мы сначала подключаем гугл-диск, затем считываем с него загруженный ранее `.csv` и сохраняем его в датафрейм `df` с которым дальше и будем работать.


3. С помощью метода `head` проверим всё ли корректно считалось.

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Да, всё гуд. Судя по первым пяти строчкам всё нормально считалось. Двигаемся дальшею

# 2. Вывести основную информацию о датасете: информацию о типах данных, число пропусков, средние значения и т.д.

1. Сначала выведем всю основную информацию методом инфо `info`

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Метод показал общее количество строк и столбцов в таблице: `891, 12`, названия всех столбцов и их типы данных `dtypes`.
Также вывел общее количество ненулевых значений в каждом столбце, например, в `Cabin` их `204`.
Ещё и общий объём памяти - `83.7 КВ`

2. Разделим все интересные нам данные.

Тип данных -> `df.types `    
*здесь у меня была прикольная ошибка, я писал со скобками `df.types()` как метод, а это же вообще не метод - это атрибут*

In [5]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


Тут мы видим, что `Survived` это выжил/не выжил, то есть 0/1 - `int`, а `Name` это имя пассажира с типом `object`.



Дальше число пропусков ->

In [6]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


Получил ичисло пустых ячеек для каждого столбца. Из `891` вычесть `204` из метода `info` - получаем число `687` пропусков для `Cabin`. Удивительно, не иначе!

Средние значения->

In [7]:
df.mean(numeric_only=True)



,0
PassengerId,446.000000
Survived,0.383838
Pclass,2.308642
Age,29.699118
SibSp,0.523008
Parch,0.381594
Fare,32.204208


Просто жесть, совсем молодые были ребята по `29.69` лет в среднем.

Всю основную информацию получили, всё гуд, идём дальше.


# 3. Посчитать процент выживаемости у каждого класса пассажиров (Pclass)

1. Здесь в одну строку справляеся с помощью группировки по `Pclass` и подсчёту среднего от `Survived`. Умножаем на 100 - получаем процент выживших.

In [8]:
my_survi_percent = df.groupby('Pclass')['Survived'].mean()*100
my_survi_percent

,Survived
Pclass,
1,62.962963
2,47.282609
3,24.236253


Пассажиров первого класса выжило значительно больше, чем пассажиров третьего - 63 против 24 процентов. Видимо всё таки в деньгах счастье.

Двигаемся дальше.

#   4. Вывести самое популярное мужское и самое популярное женское имя на корабле


Основная хитрость здесь в том, чтобы отделять Mr и Mrs и сделать столбик только с именами, а дальше на основе данных из столбца Sex решить чьё имя женское, а чьё мужское. Посчитать количество имён и вывести самое популярное можно методами `value_counts` который считает частоту каждого уникального значения и `index[]` который возвращает самое популярное.

In [9]:
def extract_first_name(passenger_name):
    try:
        # Берем часть после запятой и точки, убираем пробелы
        after_comma_dot = passenger_name.split(',')[1].split('.')[1].strip()
        # Если есть скобки - берем имя внутри, иначе - первое слово
        if '(' in after_comma_dot and ')' in after_comma_dot:
            return after_comma_dot.split('(')[1].split(')')[0].split(' ')[0]
        return after_comma_dot.split(' ')[0]
    except:
        return ''
# Применяем к столбцу Name и анализируем
df['first_name'] = df['Name'].apply(extract_first_name)
# Фильтруем мужчин и женщин
male_passengers = df[df['Sex'] == 'male']
female_passengers = df[df['Sex'] == 'female']

print('Самое популярное мужское имя:',
      male_passengers['first_name'].value_counts().index[0])
print('Самое популярное женское имя:',
      female_passengers['first_name'].value_counts().index[0])


Самое популярное мужское имя: William
Самое популярное женское имя: Anna


Самым популярным мужским именем оказался William, а женским Anna.

Всё гуд, идём дальше.

# 5. Вывести самое популярное мужское и самое популярное женское имя на корабле в каждом классе.

Здесь мы перебираем классы (1, 2, 3), далее для каждого класса фильтруем мужчин и женщин, считаем частоту их имен из столбца `first_name` (который мы ранее создали парсингом полного имени), находим самое популярное мужское и женское имя с помощью `value_counts() и index[0]` и выводим результат.

In [10]:
for passenger_class in sorted(df['Pclass'].unique()):
    class_passengers = df[df['Pclass'] == passenger_class]

    male_names = class_passengers[class_passengers['Sex'] == 'male']['first_name'].value_counts()
    female_names = class_passengers[class_passengers['Sex'] == 'female']['first_name'].value_counts()

    most_common_male = male_names.index[0]
    most_common_female = female_names.index[0]

    print(f'Класс {passenger_class}: '
          f'мужское - {most_common_male}, '
          f'женское - {most_common_female}')


Класс 1: мужское - William, женское - Margaret
Класс 2: мужское - William, женское - Elizabeth
Класс 3: мужское - William, женское - Anna


Видим, что мужские имена от класса к классу не отличаются, а Маргаритт из 1 класса оказалось поменьше чем Анн из 3 класса.

Идём дальше.

# 6. Вывести часть таблицы с пассажирами, возраст которых больше 44 лет

Здесь создаём новую таблицу с условием по столбцу `Age`, в итоге остаются только строки в которых условие > 44 выполняется.

In [11]:
my_people_44 = (df[df['Age'] > 44])
my_people_44


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,first_name
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,858,1,1,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S,Peter
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S,Margaret
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S,Sallie
873,874,0,3,"Vander Cruyssen, Mr. Victor",male,47.0,0,0,345765,9.0000,NaN,S,Victor


Снизу подписано, что всего строчек 115, значит делаем вывод, что у нас 115 пассажиров старще 44. В целом, такую же статистику нам продемонстрируют концерты Александра Розембаума.

Идём дальше.

# 7. Выведите часть таблицы с пассажирами, возраст которых меньше 44 лет и которые мужского пола

Просто добавляем условие по полу, чтобы он был строго `"male"` через оператор `&`

In [12]:
my_boys_44 = df[(df['Age'] < 44) & (df['Sex'] == 'male')]
my_boys_44

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,first_name
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.500,NaN,S,Frederick
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S,Henry
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000,NaN,S,Juozas
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000,C148,C,Karl


368 парней до 44. Стандартный сервер в майнкрафте.

Идём дальше.

# 8. Выведите количества n-местных кабин (в которых было 2, 3, 4, ... человека)

Здесь разберемся в несколько шагов. 1-й шаг берем столбец `Cabin`, убираем пустые значения с помощью метода `dropna()` из библиотеки pandas, разбиваем строки типа 27. где в Canin три каюты "C23 C25 C27", на отдельные каюты через `str.split()` и считаем частоту каждой каюты в словаре `cabin_counts` , 2-й шаг с помощью `pd.Series` и `value_counts()` группируем каюты, показывая сколько кают встречалось N раз, а затем циклом выводим результат.

In [13]:
# 1 шаг: собираем все каюты и считаем их частоту
cabin_counts = {}
for cabins in df['Cabin'].dropna().str.split():
    for cabin in cabins:
        cabin_counts[cabin] = cabin_counts.get(cabin, 0) + 1

# 2 шаг: считаем, сколько кают встречалось N раз
occupancy_counts = pd.Series(list(cabin_counts.values())).value_counts().sort_index()

# Вывод
for num_people in occupancy_counts.index:
    print(f'{num_people} - местные: {occupancy_counts[num_people]}')


1 - местные: 104
2 - местные: 44
3 - местные: 6
4 - местные: 7


1,2,3,4... Прямо как в морском бое.

Идем дальше.

# 9. Вывести количество пассажиров, у которых нет родственников на борту

Чтобы найти пассажиров без родственников, проверяем столбцы SibSp (братья/сестры, супруги) и Parch (родители/дети) — если оба равны 0, значит человек путешествовал один. ставим условие и выводим кусок таблицы из одиночек.


In [14]:
alone = df[(df['SibSp'] == 0) & (df['Parch'] == 0)]
alone

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,first_name
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Laina
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,William
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,James
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
...,...,...,...,...,...,...,...,...,...,...,...,...,...
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.0500,NaN,S,Henry
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Juozas
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Margaret
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,Karl


Дальше выведем количество строк.

In [15]:
print(f'Одиночек насчиталось: {len(alone)}')

Одиночек насчиталось: 537


Вот так. Подавляющее большинство путишествовали в одиночку.